# QMoE.rs — Colab Model & Dataset Preparation (Phase 2)

This notebook downloads a real DeepSeek-Coder-V2-Lite-Instruct model from HuggingFace,
renames tensor keys to match the Rust engine's conventions, quantizes expert weights to
ternary (2-bit, 4-per-byte packing), and packages everything for eval.

**Outputs:**
- `model.safetensors` — quantized model (~4 GB for 16B params)
- `config.json` — model hyperparameters for the Rust loader
- `wikitext2.txt` — WikiText-2 test split
- `quantize.py` — reference quantization script
- `qmoe_eval_artifacts.tar.gz` — all of the above bundled

## 1. Install Dependencies

In [ ]:
!pip install torch safetensors transformers datasets tqdm

## 2. Imports

In [ ]:
import torch
import json
import os
import tarfile
from safetensors.torch import save_file
from transformers import AutoModelForCausalLM, AutoConfig
from datasets import load_dataset

## 3. Save quantize.py to Workspace

Save a copy of the quantization script so it's included in the tarball for reference.
The actual quantization is done in-memory in this notebook (same algorithm).

In [ ]:
quantize_py = r'''
import torch
from safetensors.torch import save_file
import argparse
import os

def quantize_and_pack(weight, scale):
    scaled = weight / scale[:, None]
    quantized = torch.clamp(torch.round(scaled), -1, 1).to(torch.int32)
    mapped = quantized + 1
    out_features, in_features = mapped.shape
    assert in_features % 4 == 0
    grouped = mapped.view(out_features, in_features // 4, 4)
    packed = (grouped[..., 3] << 6) | (grouped[..., 2] << 4) | (grouped[..., 1] << 2) | grouped[..., 0]
    return packed.to(torch.uint8)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=str, default=None)
    parser.add_argument("--output", type=str, default="qmoe_packed_model.safetensors")
    args = parser.parse_args()
    print("--- Starting QMoE Offline Quantization & Packing ---")
    if args.input is None or not os.path.exists(args.input):
        print("No valid input. Generating mock weights...")
        num_layers = 2
        num_experts = 4
        hidden_dim = 64
        intermediate_dim = 128
        vocab_size = 102400
        state_dict = {}
        state_dict["embed.weight"] = torch.randn(vocab_size, hidden_dim)
        state_dict["norm.weight"] = torch.randn(hidden_dim)
        state_dict["norm.bias"] = torch.randn(hidden_dim)
        state_dict["lm_head.weight"] = torch.randn(vocab_size, hidden_dim)
        for l in range(num_layers):
            state_dict[f"layers.{l}.moe.gate.weight"] = torch.randn(num_experts, hidden_dim)
            state_dict[f"layers.{l}.self_attn.q_proj.weight"] = torch.randn(hidden_dim, hidden_dim)
            state_dict[f"layers.{l}.self_attn.k_proj.weight"] = torch.randn(hidden_dim, hidden_dim)
            state_dict[f"layers.{l}.self_attn.v_proj.weight"] = torch.randn(hidden_dim, hidden_dim)
            state_dict[f"layers.{l}.self_attn.o_proj.weight"] = torch.randn(hidden_dim, hidden_dim)
            state_dict[f"layers.{l}.input_layernorm.weight"] = torch.randn(hidden_dim)
            state_dict[f"layers.{l}.input_layernorm.bias"] = torch.randn(hidden_dim)
            state_dict[f"layers.{l}.post_attention_layernorm.weight"] = torch.randn(hidden_dim)
            state_dict[f"layers.{l}.post_attention_layernorm.bias"] = torch.randn(hidden_dim)
            for e in range(num_experts):
                state_dict[f"layers.{l}.moe.experts.{e}.gate_proj.weight"] = torch.randn(intermediate_dim, hidden_dim)
                state_dict[f"layers.{l}.moe.experts.{e}.up_proj.weight"] = torch.randn(intermediate_dim, hidden_dim)
                state_dict[f"layers.{l}.moe.experts.{e}.down_proj.weight"] = torch.randn(hidden_dim, intermediate_dim)
    else:
        print(f"Loading weights from {args.input}...")
        state_dict = torch.load(args.input)
    packed_dict = {}
    for key, tensor in state_dict.items():
        if "gate_proj.weight" in key or "up_proj.weight" in key or "down_proj.weight" in key:
            print(f"Quantizing: {key}...")
            scale = tensor.std(dim=1).clamp(min=1e-5)
            packed_weights = quantize_and_pack(tensor, scale)
            packed_dict[key] = packed_weights
            packed_dict[key.replace(".weight", ".scales")] = scale
        else:
            print(f"Keeping: {key}...")
            packed_dict[key] = tensor.to(torch.float32)
    print(f"Saving to {args.output}...")
    save_file(packed_dict, args.output)
    print("--- Done! ---")

if __name__ == "__main__":
    main()
'''

with open("quantize.py", "w") as f:
    f.write(quantize_py.strip())
print("quantize.py saved")

## 4. Download Model from HuggingFace

Downloads DeepSeek-Coder-V2-Lite-Instruct (~16B params).

**Memory note:** A 16B model in fp16 is ~32 GB. We load on CPU with `low_cpu_mem_usage=True`.
If you run out of RAM, restart with a High-RAM runtime (Colab Pro) or use T4 GPU + `device_map="auto"`.
The model uses `trust_remote_code=True` because DeepSeek has custom architecture code.

In [ ]:
MODEL_ID = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
print(f"Loading config from {MODEL_ID}...")
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"Loading model from {MODEL_ID} (this will take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map="cpu",
)
state_dict = model.state_dict()
print(f"Loaded {len(state_dict)} tensors from the model")
del model  # free memory

## 5. Rename Tensors

Map HuggingFace tensor names to the names expected by the Rust engine.

| HuggingFace key | → Rust key |
|---|---|
| `model.layers.{i}.self_attn.{q,k,v,o}_proj.weight` | `layers.{i}.self_attn.{q,k,v,o}_proj.weight` |
| `model.layers.{i}.self_attn.{q,k,v,o}_proj.bias` | `layers.{i}.self_attn.{q,k,v,o}_proj.bias` |
| `model.layers.{i}.mlp.gate.weight` | `layers.{i}.moe.gate.weight` |
| `model.layers.{i}.mlp.experts.{e}.gate_proj.weight` | `layers.{i}.moe.experts.{e}.gate_proj.weight` |
| `model.layers.{i}.mlp.experts.{e}.up_proj.weight` | `layers.{i}.moe.experts.{e}.up_proj.weight` |
| `model.layers.{i}.mlp.experts.{e}.down_proj.weight` | `layers.{i}.moe.experts.{e}.down_proj.weight` |
| `model.layers.{i}.input_layernorm.weight` | `layers.{i}.input_layernorm.weight` |
| `model.layers.{i}.post_attention_layernorm.weight` | `layers.{i}.post_attention_layernorm.weight` |
| `model.embed_tokens.weight` | `embed.weight` |
| `model.norm.weight` | `norm.weight` |
| `lm_head.weight` | `lm_head.weight` |

In [ ]:
def rename_hf_to_rust(key: str) -> str:
    if key.startswith("model.layers."):
        # Strip "model." prefix, replace "mlp" with "moe"
        new_key = key[len("model."):]
        new_key = new_key.replace(".mlp.", ".moe.")
        return new_key
    elif key == "model.embed_tokens.weight":
        return "embed.weight"
    elif key == "model.norm.weight":
        return "norm.weight"
    elif key == "model.norm.bias":
        return "norm.bias"
    # lm_head.weight, lm_head.bias, and any other keys stay as-is
    return key

renamed = {}
for k, v in state_dict.items():
    new_k = rename_hf_to_rust(k)
    if new_k != k:
        print(f"  {k[:80]}... → {new_k}")
    renamed[new_k] = v

print(f"\nRenamed {len(renamed)} tensors")
del state_dict  # free memory

## 6. Quantize Expert Weights

Quantize each expert's `gate_proj`, `up_proj`, `down_proj` weights to ternary values
({-1, 0, 1}) and pack 4 weights per uint8 byte (2 bits each).

All other weights (attention projections, gating, norms, embeddings, lm_head)
are kept in fp32.

In [ ]:
packed_dict = {}
expert_keys = ["gate_proj.weight", "up_proj.weight", "down_proj.weight"]

for key, tensor in renamed.items():
    is_expert_weight = any(ek in key for ek in expert_keys)

    if is_expert_weight:
        print(f"[quantize] {key}  shape={list(tensor.shape)}  dtype={tensor.dtype}")

        # Row-wise scaling factor
        scale = tensor.float().std(dim=1).clamp(min=1e-5)

        # Quantize to ternary [-1, 0, 1]
        scaled = tensor.float() / scale[:, None]
        quantized = torch.clamp(torch.round(scaled), -1, 1).to(torch.int32)
        mapped = quantized + 1  # [-1, 0, 1] → [0, 1, 2]

        out_f, in_f = mapped.shape
        assert in_f % 4 == 0, f"{key}: in_features {in_f} not divisible by 4"

        # Pack 4 weights per byte: [w3(2b) | w2(2b) | w1(2b) | w0(2b)]
        grouped = mapped.view(out_f, in_f // 4, 4)
        packed = (
            (grouped[..., 3] << 6)
            | (grouped[..., 2] << 4)
            | (grouped[..., 1] << 2)
            | grouped[..., 0]
        )

        packed_dict[key] = packed.to(torch.uint8)
        packed_dict[key.replace(".weight", ".scales")] = scale.to(torch.float32)
    else:
        print(f"[keep]    {key}  shape={list(tensor.shape)}  dtype={tensor.dtype}")
        packed_dict[key] = tensor.contiguous().to(torch.float32)

print(f"\nPacked dictionary: {len(packed_dict)} entries")
del renamed  # free memory

## 7. Save model.safetensors

In [ ]:
print("Saving model.safetensors...")
save_file(packed_dict, "model.safetensors")
file_size = os.path.getsize("model.safetensors") / 1e9
print(f"Saved model.safetensors ({file_size:.2f} GB)")
del packed_dict  # free memory

## 8. Generate config.json

Extract hyperparameters from the HuggingFace config and map them to the names
expected by the Rust `loader.rs::ConfigJson`.

In [ ]:
# Map HuggingFace config attributes to Rust ConfigJson keys
hf_config = {}

# Standard fields
hf_config["vocab_size"] = config.vocab_size
hf_config["hidden_size"] = config.hidden_size
hf_config["num_hidden_layers"] = config.num_hidden_layers
hf_config["intermediate_size"] = getattr(config, "moe_intermediate_size", None) or \
                                  getattr(config, "intermediate_size", None)

# DeepSeek-specific expert fields
hf_config["num_experts"] = getattr(config, "n_routed_experts", None) or \
                           getattr(config, "num_experts", None) or \
                           getattr(config, "num_local_experts", None)
hf_config["top_k"] = getattr(config, "num_experts_per_tok", None) or \
                      getattr(config, "top_k", None) or 1

# hidden_dim defaults to hidden_size for compatibility
hf_config["hidden_dim"] = config.hidden_size

print("Config extracted:")
for k, v in hf_config.items():
    print(f"  {k}: {v}")

with open("config.json", "w") as f:
    json.dump(hf_config, f, indent=2)
print("\nSaved config.json")
del config

## 9. Download WikiText-2 Dataset

Download the WikiText-2 test split from HuggingFace datasets.

In [ ]:
print("Downloading WikiText-2 test split...")
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
text = "\n".join(dataset["text"])
with open("wikitext2.txt", "w", encoding="utf-8") as f:
    f.write(text)
print(f"Saved wikitext2.txt ({len(text)} chars, {len(dataset)} lines)")
del dataset, text

## 10. Package Tarball

Combine all artifacts into a single tarball for download.

In [ ]:
artifact_files = ["model.safetensors", "config.json", "wikitext2.txt", "quantize.py"]

print("Creating qmoe_eval_artifacts.tar.gz...")
with tarfile.open("qmoe_eval_artifacts.tar.gz", "w:gz") as tar:
    for fname in artifact_files:
        if os.path.exists(fname):
            tar.add(fname, arcname=fname)
            fsize = os.path.getsize(fname)
            print(f"  Added {fname} ({fsize / 1e6:.1f} MB)")
        else:
            print(f"  WARNING: {fname} not found, skipping")

tarball_size = os.path.getsize("qmoe_eval_artifacts.tar.gz") / 1e9
print(f"\nCreated qmoe_eval_artifacts.tar.gz ({tarball_size:.2f} GB)")

## 11. Summary

All artifacts are ready. Download `qmoe_eval_artifacts.tar.gz` from Colab's file browser.

**Next steps on your local machine:**

```bash
# Extract
tar xzf qmoe_eval_artifacts.tar.gz -C ./eval_artifacts/

# Run perplexity eval (Phase 3)
cargo run --release --bin eval -- \
    --model eval_artifacts/model.safetensors \
    --config eval_artifacts/config.json \
    --tokenizer resources/tokenizer.json \
    --dataset eval_artifacts/wikitext2.txt \
    --block-size 128 --stride 64

# Run integration tests (Phase 4)
cargo test --release
```

In [ ]:
print("Phase 2 complete! Download qmoe_eval_artifacts.tar.gz from the file browser.")